# Part 5: Cluster pixels

In the following notebook you will begin the process of phenotyping cells by using a self-organizing map algorithm to cluster the pixels.  This is the "Pixie" project from the ark-analysis repo https://github.com/angelolab/ark-analysis/tree/main

## 1. Import packages. 
*This must be done every time the notebook is started or restarted.

In [ ]:
# import required packages

import os
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from alpineer import io_utils, load_utils
from matplotlib import rc_file_defaults
import pandas as pd
import numpy as np
import pyFlowSOM
import seaborn as sns
from matplotlib.colors import ListedColormap
import tifffile as tiff
from tqdm.notebook import tqdm
import Kview

# from ark.phenotyping import (pixel_cluster_utils, pixel_meta_clustering,
#                              pixel_som_clustering, pixie_preprocessing)
# from ark.utils import data_utils
# from ark.utils import plot_utils
# from ark.utils.metacluster_remap_gui import (MetaClusterGui,
#                                              colormap_helper,
#                                              metaclusterdata_from_files)

## 2. Define directory paths. 
*This must be done every time the notebook is started or restarted.

In [2]:
base_dir = "C:\\Users\\smith6jt"

In [3]:
seg_dir = os.path.join(base_dir, "KINTSUGI", "data", "1904_CC2B_Segmentation")
proc_dir = seg_dir.replace('_Segmentation', '_Processed')
clus_dir = seg_dir.replace('_Segmentation', '_Clustering')
print(f"Segmentation folder is {seg_dir}.")
print(f"Processed folder is {proc_dir}.")
print(f"Clustering folder is {clus_dir}.")

Segmentation folder is C:\Users\smith6jt\KINTSUGI\data\1904_CC2B_Segmentation.
Processed folder is C:\Users\smith6jt\KINTSUGI\data\1904_CC2B_Processed.
Clustering folder is C:\Users\smith6jt\KINTSUGI\data\1904_CC2B_Clustering.


In [4]:
os.makedirs(clus_dir, exist_ok=True)

## 4. Pixel clustering

### 4.1 Visualize pixel SOM performance

Test parameters:  Choose fov and channel to visualize, channels to include, number of nodes (this will be xdim x ydim), and the number of training epochs (num_passes).  The learning rates are a more advanced parameter.  Seed is kept constant if reproducible results are important.

In [43]:
intensity_path = os.path.join(seg_dir, 'marker_intensities.csv')
intensity_df = pd.read_csv(intensity_path)
intensity_df[channels].head()

,cell_CD20_mean,cell_CD3e_mean,cell_CD68_mean
0,0.0,0.0,161.472805
1,0.0,0.0,366.400286
2,0.0,0.0,152.716364
3,0.0,0.0,467.305909
4,0.0,0.0,99.764215


In [18]:
image = tiff.imread(os.path.join(proc_dir,  f"{channel_vis}.tif"))
nan_mask = np.isnan(image)
nan_indices = np.where(nan_mask)[0]
find_nan_df = intensity_df.loc[intensity_df.isna().any(axis=1)]
nan_indices

array([], dtype=int64)

In [54]:
def visualize_som(pixel_data: None, 
                  channels: list,
                         image: str,
                         channel_vis: str,
                         num_nodes: int = 10, 
                         num_passes: int = 5,
                         lr_start: float = 0.05,
                         lr_end: float = 0.01,
                         live_plots: bool = False):
    
    train_data = pixel_data[channels].copy()
    train_data['cell_centroid_0'] = pixel_data['cell_centroid_0']
    train_data['cell_centroid_1'] = pixel_data['cell_centroid_1']
    train_array = pixel_data[channels].values.astype(np.float64)
    colors = sns.color_palette("tab20", n_colors=num_nodes)
    node_cmap = ListedColormap(colors)

    pbar_filesave = tqdm(total=100, unit="Percent",
                    bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]',
                    colour="green", position=0, leave=True)

    scatter_frames = []
    for rlen_iter in range(1, num_passes + 1):
        # print(f"Training for rlen={rlen_iter}")
        som = pyFlowSOM.som(train_array, num_nodes//2, num_nodes//2, rlen_iter, alpha_range=(lr_start, lr_end), seed=627)
        clusters, dist = pyFlowSOM.map_data_to_nodes(som, train_array)

        train_data['cluster'] = clusters
        df_mean = train_data.groupby(['cluster']).mean()
        df_mean = df_mean.drop(columns=['cell_centroid_0', 'cell_centroid_1'])
        fig, axes = plt.subplots(1, 3, figsize=(21, 7))
        im = axes[0].imshow(image, vmax=np.percentile(image, 99.9), cmap='turbo')
        fig.colorbar(im, shrink=0.5, ax=axes[0], cmap=node_cmap)
        axes[0].set_title(f'Original Image: {channel_vis}')
        im = axes[1].scatter(train_data['cell_centroid_0'], train_data['cell_centroid_1'], 
                                c=train_data['cluster'],
                                cmap=node_cmap, s=1)
        axes[1].invert_yaxis()
        axes[1].set_aspect(image.shape[1] / image.shape[0])
        axes[1].set_title(f'SOM Iteration {rlen_iter}')
        cbar = fig.colorbar(im, shrink=0.8, ax=axes[1], cmap=node_cmap, aspect=30, pad=0.02)
        ticks = np.linspace(0, num_nodes-1, num_nodes) 
        ticks = np.arange(num_nodes)
        cbar.set_ticks(ticks)
        axes[1].set_title(f"SOM Visualization for rlen=(rlen={rlen_iter})")

        g=sns.clustermap(df_mean, z_score=3, cmap="vlag", center=0, 
                        yticklabels=True, cbar=node_cmap)
        heatmap_data = g.data2d
        plt.close(g.figure)
        im = sns.heatmap(data=heatmap_data, 
                ax=axes[2],
                cmap="vlag",
                center=0,
                xticklabels=g.data2d.columns,
                yticklabels=g.data2d.index)
        
        axes[2].set_aspect(image.shape[1] / image.shape[0])
        axes[2].set_title(f'Cluster heatmap for rlen={rlen_iter}')
        
        plt.setp(axes[2].get_xticklabels(), rotation=90)
        plt.setp(axes[2].get_yticklabels(), rotation=0)
        fig.tight_layout()

        canvas = fig.canvas
        canvas.draw()
        width, height = fig.get_size_inches() * fig.dpi
        buf = np.frombuffer(canvas.buffer_rgba(), dtype=np.uint8)
        buf.shape = (int(height), int(width), 4)
        scatter_frame = buf[:, :, :3]
        
        scatter_frames.append(scatter_frame)
        if live_plots:
            plt.show()
        plt.close(fig)
        pbar_filesave.update(100 / (num_passes))

    pbar_filesave.close()
    scatter_stack = np.stack(scatter_frames, axis=0)
    return scatter_stack


In [57]:
channel_vis = 'CD3e'
channels = ['cell_CD20_mean','cell_CD3e_mean', 'cell_CD68_mean']
xdim = 3
ydim = 3
num_passes = 3
lr_start = 0.05
lr_end = 0.01
file=os.path.join(seg_dir, 'marker_intensities.csv')
pixel_data = pd.read_csv(file)
num_nodes = xdim * ydim
image = tiff.imread(os.path.join(proc_dir,  f"{channel_vis}.tif"))

stack = visualize_som(
    pixel_data=pixel_data,
    channels=channels,
    image = image,
    channel_vis = channel_vis,
    num_nodes = num_nodes,
    num_passes=num_passes,
    lr_start=lr_start,
    lr_end=lr_end,
    live_plots=False
)
import stackview
stackview.slice(stack)

  0%|          | 0/100 [00:00<?]

In [ ]:
max_k = 10
cap = 3

# run hierarchical clustering using average pixel SOM cluster expression
pixel_cc = pixel_meta_clustering.pixel_consensus_cluster(
    fovs,
    channels,
    base_dir,
    max_k=max_k,
    cap=cap,
    data_dir=pixel_data_dir,
    pc_chan_avg_som_cluster_name=pc_chan_avg_som_cluster_name,
    multiprocess=True,
    overwrite=False,
    batch_size=batch_size
)

# generate the meta cluster summary files
pixel_meta_clustering.generate_meta_avg_files(
    fovs,
    channels,
    base_dir,
    pixel_cc,
    data_dir=pixel_data_dir,
    overwrite=False,
    pc_chan_avg_som_cluster_name=pc_chan_avg_som_cluster_name,
    pc_chan_avg_meta_cluster_name=pc_chan_avg_meta_cluster_name
)

## 5. Visualize results

### 5.1: Interactive adjustments to relabel pixel meta clusters
The visualization shows the z-scored average channel expression per pixel SOM and meta cluster. The heatmaps are faceted by pixel SOM clusters on the left and pixel meta clusters on the right.


Quickstart
- **Select**: Left Click
- **Remap**: **New metacluster** button or Right Click
- **Edit Metacluster Name**: Textbox at bottom right of the heatmaps.

Selection and remapping details
- To select a SOM cluster, click on its respective position in the **selected** bar. Click on it again to deselect.
- To select a meta cluster, click on its corresponding color in the **metacluster** bar. Click on it again to deselect.
- To remap the selected clusters, click the **New metacluster** button (alternatively, right click anywhere). Note that remapping an entire metacluster deletes it.
- To clear the selected SOM/meta clusters, use the **Clear Selection** button.
- **After remapping a meta cluster, make sure to deselect the newly created one to prevent unwanted combinations.**

Other features and notes
- You will likely need to zoom out to see the entire visualization. To toggle Zoom, use Ctrl -/Ctrl + on Windows or ⌘ +/⌘ - on Mac.
- The bars at the top show the number of pixels in each SOM cluster.
- The text box at the bottom right allows you to rename a particular meta cluster. This can be useful as remapping may cause inconsistent numbering. **You cannot use the same name for different meta clusters; doing so will cause the next step to fail.**
- Adjust the z-score limit using the slider on the bottom left to adjust your dynamic range.
- When meta clusters are combined or a meta cluster is renamed, the change is immediately saved to `pixel_meta_cluster_remap_name`.
- You won't be able to advance in the notebook until you've clicked `New metacluster` or renamed a meta cluster at least once. If you don't want to make changes, just click `New metacluster` to trigger a save before continuing.

In [ ]:
%matplotlib widget
rc_file_defaults()
plt.ion()

pixel_mcd = metaclusterdata_from_files(
    os.path.join(base_dir, pc_chan_avg_som_cluster_name),
    cluster_type='pixel'
)
pixel_mcd.output_mapping_filename = os.path.join(base_dir, pixel_meta_cluster_remap_name)
pixel_mcg = MetaClusterGui(pixel_mcd, width=15)

Relabel the pixel meta clusters using the mapping, and recompute the meta cluster average files with the new meta cluster names.

In [ ]:
# rename the meta cluster values in the pixel dataset
pixel_meta_clustering.apply_pixel_meta_cluster_remapping(
    fovs,
    channels,
    base_dir,
    pixel_data_dir,
    pixel_meta_cluster_remap_name,
    multiprocess=multiprocess,
    batch_size=batch_size
)

# recompute the mean channel expression per meta cluster and apply these new names to the SOM cluster average data
pixel_meta_clustering.generate_remap_avg_files(
    fovs,
    channels,
    base_dir,
    pixel_data_dir,
    pixel_meta_cluster_remap_name,
    pc_chan_avg_som_cluster_name,
    pc_chan_avg_meta_cluster_name
)

Generate the color scheme returned by the interactive reclustering process. This will be for visualizing the pixel phenotype maps.

In [23]:
raw_cmap, _ = colormap_helper.generate_meta_cluster_colormap_dict(
    pixel_mcd.output_mapping_filename,
    pixel_mcg.im_cl.cmap
)

### 5.2: Generate pixel phenotype maps

Generate pixel phenotype maps, in which each pixel in the image corresponds to its pixel meta cluster. Select a small subset of your FOVs to view within this notebook. Or if you wish to generate and save a significant amount of FOVs, the masks will be created and saved in batches.

Files will be written as `{fov_name}_pixel_mask.tiff` in `pixel_output_dir`

In [24]:
# select fovs to display
subset_pixel_fovs = ['fov0', 'fov1', 'fov2', 'fov3', 'fov4']
# , 'fov5', 'fov6', 'fov7', 'fov8', 'fov9', 'fov10', 'fov11', 'fov12', 'fov13', 'fov14', 'fov15', 'fov16', 'fov17', 'fov18', 'fov19', 'fov20', 'fov21','fov22', 'fov23']


In [ ]:
# define the path to the channel file
if img_sub_folder is None:
    chan_file = os.path.join(
        io_utils.list_files(os.path.join(tiff_dir, fovs[0]), substrs=['.tif'])[0]
    )
else:
    chan_file = os.path.join(
        img_sub_folder, io_utils.list_files(os.path.join(tiff_dir, fovs[0], img_sub_folder), substrs=['.tif'])[0]
    )

# generate and save the pixel cluster masks for each fov in subset_pixel_fovs
data_utils.generate_and_save_pixel_cluster_masks(
    fovs=subset_pixel_fovs,
    base_dir=base_dir,
    save_dir=os.path.join(base_dir, pixel_output_dir),
    tiff_dir=tiff_dir,
    chan_file=chan_file,
    pixel_data_dir=pixel_data_dir,
    cluster_id_to_name_path=os.path.join(base_dir, pixel_meta_cluster_remap_name),
    pixel_cluster_col='pixel_meta_cluster',
    sub_dir='pixel_masks',
    name_suffix='_pixel_mask',
)

Save the colored pixel masks for each FOV in `subset_pixel_fovs`.

In [ ]:
plot_utils.save_colored_masks(
    fovs=subset_pixel_fovs,
    mask_dir=os.path.join(base_dir, pixel_output_dir, "pixel_masks"),
    save_dir=os.path.join(base_dir, pixel_output_dir, "pixel_mask_colored"),
    cluster_id_to_name_path=os.path.join(base_dir, pixel_meta_cluster_remap_name),
    metacluster_colors=raw_cmap,
    cluster_type="pixel"
)

Load a subset of the pixel cluster masks that you would like to preview. If the dimensions or text size of the plot need to be adjusted for optimal viewing, change the `figsize` and `dpi` parameters.

In [ ]:

for pixel_fov in subset_pixel_fovs:
    pixel_cluster_mask = load_utils.load_imgs_from_dir(
        data_dir=os.path.join(base_dir, pixel_output_dir, "pixel_masks"),
        files=[pixel_fov + "_pixel_mask.tiff"],
        trim_suffix="_pixel_mask",
        match_substring="_pixel_mask",
        xr_dim_name="pixel_mask",
        xr_channel_names=None,
    )

    plot_utils.plot_pixel_cell_cluster(
        pixel_cluster_mask,
        [pixel_fov],
        os.path.join(base_dir, pixel_meta_cluster_remap_name),
        metacluster_colors=raw_cmap,
        figsize=(8, 8),
        dpi=100
    )